# 🎓 Professional Mentorship System with Real Sri Lanka Alumni Data

This notebook integrates your existing datasets (mentors, mentees, alumni) and automatically generates the missing components for a complete mentorship platform.

**Your Available Datasets:**
- mentors.csv
- mentees.csv  
- sri_lanka_alumni_dataset_6000_cleaned_synthetic.csv

**Auto-Generated Datasets:**
- mentorship_matching.csv
- communication_logs.csv
- api_integration_logs.csv
- privacy_workflow.csv

## 📦 Install Dependencies & Setup

In [3]:
# Install required packages
!pip install pandas numpy scikit-learn matplotlib seaborn google-colab > /dev/null 2>&1

import pandas as pd
import numpy as np
import random
import json
import hashlib
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

try:
    from google.colab import drive, files
    drive.mount('/content/drive')
    print("✅ Google Drive connected successfully!")
    DRIVE_PATH = '/content/drive/MyDrive/mentorship_data/'
    import os
    os.makedirs(DRIVE_PATH, exist_ok=True)
    print(f"📁 Data directory: {DRIVE_PATH}")
except:
    print("🔄 Using local storage...")
    DRIVE_PATH = '/content/'

print("🎯 Setup complete!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive connected successfully!
📁 Data directory: /content/drive/MyDrive/mentorship_data/
🎯 Setup complete!


## 📊 Load Your Existing Datasets

In [4]:
# Define dataset paths - update these to match your actual file locations
existing_datasets = {
    'mentors': f'{DRIVE_PATH}mentors.csv',
    'mentees': f'{DRIVE_PATH}mentees.csv',
    'alumni': f'{DRIVE_PATH}sri_lanka_alumni_dataset_6000_cleaned_synthetic.csv'
}

# Generated datasets paths
generated_datasets = {
    'matching': f'{DRIVE_PATH}mentorship_matching.csv',
    'communication': f'{DRIVE_PATH}communication_logs.csv',
    'api': f'{DRIVE_PATH}api_integration_logs.csv',
    'privacy': f'{DRIVE_PATH}privacy_workflow.csv'
}

# Load existing datasets
print("📁 Loading your existing datasets...")

try:
    df_mentors = pd.read_csv(existing_datasets['mentors'])
    print(f"✅ Mentors loaded: {len(df_mentors)} records")
except FileNotFoundError:
    print("❌ mentors.csv not found - will generate synthetic data")
    df_mentors = None

try:
    df_mentees = pd.read_csv(existing_datasets['mentees'])
    print(f"✅ Mentees loaded: {len(df_mentees)} records")
except FileNotFoundError:
    print("❌ mentees.csv not found - will generate synthetic data")
    df_mentees = None

try:
    df_alumni = pd.read_csv(existing_datasets['alumni'])
    print(f"✅ Alumni loaded: {len(df_alumni)} records")
except FileNotFoundError:
    print("❌ alumni dataset not found")
    df_alumni = None

print("\n📋 Dataset Status:")
print(f"  • Mentors: {'Loaded' if df_mentors is not None else 'Will Generate'}")
print(f"  • Mentees: {'Loaded' if df_mentees is not None else 'Will Generate'}")
print(f"  • Alumni: {'Loaded' if df_alumni is not None else 'Not Available'}")

📁 Loading your existing datasets...
✅ Mentors loaded: 5000 records
✅ Mentees loaded: 5000 records
✅ Alumni loaded: 6000 records

📋 Dataset Status:
  • Mentors: Loaded
  • Mentees: Loaded
  • Alumni: Loaded


In [5]:
# Display sample of your alumni data
if df_alumni is not None:
    print("🎓 Sample Alumni Data:")
    print(df_alumni.head(10))

    print("\n📊 Alumni Data Info:")
    print(f"  • Total Alumni: {len(df_alumni)}")
    print(f"  • Columns: {list(df_alumni.columns)}")

    # University distribution
    if 'University' in df_alumni.columns:
        print("\n🏫 University Distribution:")
        uni_counts = df_alumni['University'].value_counts().head(10)
        for uni, count in uni_counts.items():
            print(f"  • {uni}: {count}")

    # Degree programs
    if 'Degree Program' in df_alumni.columns:
        print("\n📚 Degree Programs (Top 10):")
        degree_counts = df_alumni['Degree Program'].value_counts().head(10)
        for degree, count in degree_counts.items():
            print(f"  • {degree}: {count}")
else:
    print("❌ No alumni data available")

🎓 Sample Alumni Data:
                        Timestamp                         Full Name   \
0  2026/05/05 1:45:18 PM GMT+5:30                Thanuja Samaraweera   
1  2026/05/05 1:49:40 PM GMT+5:30          Jesintha Kawindi Rangalla   
2  2026/05/05 1:51:46 PM GMT+5:30               Thishmi Gunawardhana   
3  2026/05/05 1:58:17 PM GMT+5:30       B.Thanushi Amanda Basnayake    
4  2026/05/05 2:01:30 PM GMT+5:30       Nethmini Dilshara Rathnayaka   
5  2026/05/05 2:07:34 PM GMT+5:30                            Dasith    
6  2026/05/05 2:35:45 PM GMT+5:30               Rukshika Madhushani    
7  2026/05/05 2:59:31 PM GMT+5:30             S.M.D Dileesha Lakshan   
8  2026/05/05 2:59:56 PM GMT+5:30  Sellappulige Praveen Dilshan Rosa   
9  2026/05/05 3:33:00 PM GMT+5:30                  Jihan jayawardane   

                           University    \
0                                 Other   
1  Sabaragamuwa University of Sri Lanka   
2  Sabaragamuwa University of Sri Lanka   
3  Sabaragamu

## 🔧 Process and Enhance Existing Data

In [6]:
# Enhance mentors dataset if it exists
if df_mentors is not None:
    print("🔧 Enhancing mentors dataset...")

    # Add missing columns if needed
    required_mentor_columns = [
        'mentor_id', 'name', 'email', 'gender', 'university', 'industry_role',
        'domain', 'experience_years', 'skills', 'location', 'linkedin_profile',
        'availability_hours', 'mentoring_style', 'languages', 'verification_level',
        'combined_verification_score', 'response_rate', 'average_rating'
    ]

    # Add missing columns with default values
    for col in required_mentor_columns:
        if col not in df_mentors.columns:
            if col == 'email':
                df_mentors[col] = df_mentors['name'].str.replace(' ', '_').str.lower() + '@university.edu'
            elif col == 'linkedin_profile':
                df_mentors[col] = 'https://linkedin.com/in/' + df_mentors['name'].str.replace(' ', '_').str.lower()
            elif col == 'verification_level':
                df_mentors[col] = np.random.choice(['Bronze', 'Silver', 'Gold'], len(df_mentors))
            elif col == 'combined_verification_score':
                df_mentors[col] = np.random.uniform(60, 95, len(df_mentors)).round(2)
            elif col == 'response_rate':
                df_mentors[col] = np.random.uniform(0.7, 1.0, len(df_mentors)).round(2)
            elif col == 'average_rating':
                df_mentors[col] = np.random.uniform(3.5, 5.0, len(df_mentors)).round(2)
            elif col == 'availability_hours':
                df_mentors[col] = np.random.randint(2, 20, len(df_mentors))
            elif col == 'experience_years':
                df_mentors[col] = np.random.randint(2, 20, len(df_mentors))
            elif col == 'languages':
                df_mentors[col] = np.random.choice(['English', 'English,Sinhala', 'English,Tamil'], len(df_mentors))
            elif col == 'mentoring_style':
                df_mentors[col] = np.random.choice(['formal', 'casual', 'structured', 'flexible'], len(df_mentors))
            else:
                df_mentors[col] = 'Default'

    print(f"✅ Mentors dataset enhanced: {len(df_mentors)} records")
    print(f"  • Columns: {list(df_mentors.columns)}")
else:
    print("❌ No mentors dataset to enhance")

🔧 Enhancing mentors dataset...
✅ Mentors dataset enhanced: 5000 records
  • Columns: ['mentor_id', 'name', 'gender', 'university', 'industry_role', 'domain', 'experience_years', 'skills', 'location', 'document_score', 'interview_score', 'verification_level', 'email', 'linkedin_profile', 'availability_hours', 'mentoring_style', 'languages', 'combined_verification_score', 'response_rate', 'average_rating']


In [7]:
# Enhance mentees dataset if it exists
if df_mentees is not None:
    print("🔧 Enhancing mentees dataset...")

    # Add missing columns if needed
    required_mentee_columns = [
        'student_id', 'name', 'email', 'gender', 'university', 'degree_program',
        'year_of_study', 'gpa', 'interests', 'career_goal', 'skill_level', 'location',
        'preferred_mentor_location', 'learning_preference', 'mentorship_goals',
        'preferred_communication_style', 'languages', 'available_hours'
    ]

    # Add missing columns with default values
    for col in required_mentee_columns:
        if col not in df_mentees.columns:
            if col == 'email':
                df_mentees[col] = df_mentees['name'].str.replace(' ', '_').str.lower() + '@student.edu'
            elif col == 'gpa':
                df_mentees[col] = np.random.uniform(2.5, 4.0, len(df_mentees)).round(2)
            elif col == 'year_of_study':
                df_mentees[col] = np.random.randint(1, 4, len(df_mentees))
            elif col == 'available_hours':
                df_mentees[col] = np.random.randint(1, 10, len(df_mentees))
            elif col == 'languages':
                df_mentees[col] = np.random.choice(['English', 'English,Sinhala', 'English,Tamil'], len(df_mentees))
            elif col == 'skill_level':
                df_mentees[col] = np.random.choice(['Beginner', 'Intermediate', 'Advanced'], len(df_mentees))
            elif col == 'preferred_mentor_location':
                df_mentees[col] = np.random.choice(['any', 'same_city', 'same_university', 'remote_only'], len(df_mentees))
            elif col == 'learning_preference':
                df_mentees[col] = np.random.choice(['visual', 'auditory', 'kinesthetic', 'reading'], len(df_mentees))
            elif col == 'mentorship_goals':
                df_mentees[col] = np.random.choice(['career_guidance', 'skill_development', 'networking', 'research_guidance', 'job_search'], len(df_mentees))
            elif col == 'preferred_communication_style':
                df_mentees[col] = np.random.choice(['formal', 'casual', 'structured', 'flexible'], len(df_mentees))
            else:
                df_mentees[col] = 'Default'

    print(f"✅ Mentees dataset enhanced: {len(df_mentees)} records")
    print(f"  • Columns: {list(df_mentees.columns)}")
else:
    print("❌ No mentees dataset to enhance")

🔧 Enhancing mentees dataset...
✅ Mentees dataset enhanced: 5000 records
  • Columns: ['student_id', 'name', 'gender', 'university', 'degree_program', 'year_of_study', 'interests', 'career_goal', 'skill_level', 'location', 'email', 'gpa', 'preferred_mentor_location', 'learning_preference', 'mentorship_goals', 'preferred_communication_style', 'languages', 'available_hours']


In [8]:
# Process alumni data to create additional mentors
if df_alumni is not None:
    print("🎓 Processing alumni data to create additional mentors...")

    # Create mentors from alumni
    alumni_mentors = []
    domains = ["Software Engineering", "AI/ML", "Cybersecurity", "Data Science", "Networking", "Business IT"]
    locations = ["Colombo", "Kandy", "Galle", "Gampaha", "Kurunegala"]
    skills_pool = ["Python", "Java", "Machine Learning", "Cybersecurity", "Data Analysis", "Cloud Computing"]

    def random_skills():
        return ",".join(random.sample(skills_pool, 3))

    for idx, row in df_alumni.iterrows():
        # Determine experience based on graduation year
        try:
            grad_year = int(row.get('Year of Graduation', 2020))
            experience = max(1, 2026 - grad_year)
        except:
            experience = random.randint(1, 10)

        # Map degree program to domain
        degree = str(row.get('Degree Program', '')).lower()
        if any(keyword in degree for keyword in ['it', 'computing', 'software', 'computer']):
            domain = random.choice(['Software Engineering', 'Data Science', 'AI/ML'])
        elif any(keyword in degree for keyword in ['business', 'management', 'admin']):
            domain = 'Business IT'
        elif any(keyword in degree for keyword in ['math', 'statistic', 'research']):
            domain = 'Data Science'
        else:
            domain = random.choice(domains)

        alumni_mentors.append({
            'mentor_id': f"ALUMNI_M{idx}",
            'name': row.get('Full Name', f'Alumni_{idx}'),
            'email': f"alumni{idx}@alumni.edu",
            'gender': random.choice(['Male', 'Female']),
            'university': row.get('University', 'Other'),
            'industry_role': random.choice(['Software Engineer', 'Data Scientist', 'Consultant', 'Product Manager', 'Research Scientist']),
            'domain': domain,
            'experience_years': experience,
            'skills': random_skills(),
            'location': random.choice(locations),
            'linkedin_profile': f'https://linkedin.com/in/alumni{idx}',
            'availability_hours': random.randint(2, 20),
            'mentoring_style': random.choice(['formal', 'casual', 'structured', 'flexible']),
            'languages': random.choice(['English', 'English,Sinhala', 'English,Tamil', 'English,Sinhala,Tamil']),
            'verification_level': random.choice(['Bronze', 'Silver', 'Gold']),
            'combined_verification_score': round(random.uniform(60, 95), 2),
            'response_rate': round(random.uniform(0.7, 1.0), 2),
            'average_rating': round(random.uniform(3.5, 5.0), 2),
            'mentorship_capacity': random.randint(1, 5),
            'current_mentees': 0,
            'total_sessions': random.randint(0, 100),
            'alumni_source': True
        })

    df_alumni_mentors = pd.DataFrame(alumni_mentors)
    print(f"✅ Created {len(df_alumni_mentors)} mentors from alumni data")

    # Combine with existing mentors
    if df_mentors is not None:
        df_mentors = pd.concat([df_mentors, df_alumni_mentors], ignore_index=True)
        print(f"📊 Combined mentors: {len(df_mentors)} total records")
    else:
        df_mentors = df_alumni_mentors
        print(f"📊 Using alumni as mentors: {len(df_mentors)} records")
else:
    print("❌ No alumni data available")

🎓 Processing alumni data to create additional mentors...
✅ Created 6000 mentors from alumni data
📊 Combined mentors: 11000 total records


## 🔄 Generate Missing Synthetic Data

In [9]:
# Generate synthetic mentors if still needed
if df_mentors is None or len(df_mentors) < 1000:
    print("🔄 Generating additional synthetic mentors...")

    universities = [
        "University of Colombo", "University of Moratuwa",
        "SLIIT", "IIT Sri Lanka", "NSBM",
        "University of Kelaniya", "University of Peradeniya",
        "Sabaragamuwa University of Sri Lanka", "University of Ruhuna",
        "Eastern University, Sri Lanka", "Wayamba University of Sri Lanka"
    ]

    domains = ["Software Engineering", "AI/ML", "Cybersecurity", "Data Science", "Networking", "Business IT"]
    locations = ["Colombo", "Kandy", "Galle", "Gampaha", "Kurunegala"]
    skills_pool = ["Python", "Java", "Machine Learning", "Cybersecurity", "Data Analysis", "Cloud Computing"]

    def random_skills():
        return ",".join(random.sample(skills_pool, 3))

    # Determine how many to generate
    target_count = 5000
    current_count = len(df_mentors) if df_mentors is not None else 0
    needed = target_count - current_count

    synthetic_mentors = []
    for i in range(needed):
        synthetic_mentors.append({
            'mentor_id': f"SYNTH_M{i}",
            'name': f"Synthetic_Mentor_{i}",
            'email': f"synthetic{i}@university.edu",
            'gender': random.choice(['Male', 'Female']),
            'university': random.choice(universities),
            'industry_role': random.choice(['Lecturer', 'Software Engineer', 'Data Scientist', 'Consultant', 'Product Manager']),
            'domain': random.choice(domains),
            'experience_years': random.randint(2, 20),
            'skills': random_skills(),
            'location': random.choice(locations),
            'linkedin_profile': f'https://linkedin.com/in/synthetic{i}',
            'availability_hours': random.randint(2, 20),
            'mentoring_style': random.choice(['formal', 'casual', 'structured', 'flexible']),
            'languages': random.choice(['English', 'English,Sinhala', 'English,Tamil']),
            'verification_level': random.choice(['Bronze', 'Silver', 'Gold']),
            'combined_verification_score': round(random.uniform(60, 95), 2),
            'response_rate': round(random.uniform(0.7, 1.0), 2),
            'average_rating': round(random.uniform(3.5, 5.0), 2),
            'mentorship_capacity': random.randint(1, 5),
            'current_mentees': 0,
            'total_sessions': random.randint(0, 100),
            'alumni_source': False
        })

    df_synthetic_mentors = pd.DataFrame(synthetic_mentors)

    if df_mentors is not None:
        df_mentors = pd.concat([df_mentors, df_synthetic_mentors], ignore_index=True)
    else:
        df_mentors = df_synthetic_mentors

    print(f"✅ Generated {len(df_synthetic_mentors)} synthetic mentors")
    print(f"📊 Total mentors: {len(df_mentors)}")
else:
    print(f"✅ Sufficient mentors available: {len(df_mentors)}")

✅ Sufficient mentors available: 11000


In [10]:
# Generate synthetic mentees if needed
if df_mentees is None or len(df_mentees) < 1000:
    print("🔄 Generating synthetic mentees...")

    universities = [
        "University of Colombo", "University of Moratuwa",
        "SLIIT", "IIT Sri Lanka", "NSBM",
        "University of Kelaniya", "University of Peradeniya"
    ]

    domains = ["Software Engineering", "AI/ML", "Cybersecurity", "Data Science", "Networking", "Business IT"]
    locations = ["Colombo", "Kandy", "Galle", "Gampaha", "Kurunegala"]

    target_count = 5000
    current_count = len(df_mentees) if df_mentees is not None else 0
    needed = target_count - current_count

    synthetic_mentees = []
    for i in range(needed):
        synthetic_mentees.append({
            'student_id': f"SYNTH_S{i}",
            'name': f"Synthetic_Student_{i}",
            'email': f"synthetic_student{i}@university.edu",
            'gender': random.choice(['Male', 'Female']),
            'university': random.choice(universities),
            'degree_program': random.choice(['IT', 'CS', 'SE', 'Data Science', 'Cybersecurity']),
            'year_of_study': random.randint(1, 4),
            'gpa': round(random.uniform(2.5, 4.0), 2),
            'interests': random.choice(domains),
            'career_goal': random.choice(['Software Engineer', 'Data Scientist', 'Security Analyst', 'Lecturer']),
            'skill_level': random.choice(['Beginner', 'Intermediate', 'Advanced']),
            'location': random.choice(locations),
            'preferred_mentor_location': random.choice(['any', 'same_city', 'same_university', 'remote_only']),
            'learning_preference': random.choice(['visual', 'auditory', 'kinesthetic', 'reading']),
            'mentorship_goals': random.choice(['career_guidance', 'skill_development', 'networking']),
            'preferred_communication_style': random.choice(['formal', 'casual', 'structured', 'flexible']),
            'languages': random.choice(['English', 'English,Sinhala', 'English,Tamil']),
            'available_hours': random.randint(1, 10)
        })

    df_synthetic_mentees = pd.DataFrame(synthetic_mentees)

    if df_mentees is not None:
        df_mentees = pd.concat([df_mentees, df_synthetic_mentees], ignore_index=True)
    else:
        df_mentees = df_synthetic_mentees

    print(f"✅ Generated {len(df_synthetic_mentees)} synthetic mentees")
    print(f"📊 Total mentees: {len(df_mentees)}")
else:
    print(f"✅ Sufficient mentees available: {len(df_mentees)}")

✅ Sufficient mentees available: 5000


## 🤝 Generate Matching Data

In [11]:
# Generate mentorship matching data
print("🤝 Generating mentorship matching data...")

def calculate_compatibility_score(mentor, mentee):
    score = 0
    weights = {
        'domain_match': 0.25,
        'skills_alignment': 0.20,
        'experience_match': 0.15,
        'location_preference': 0.10,
        'communication_style': 0.10,
        'language_match': 0.10,
        'availability_match': 0.10
    }

    # Domain matching
    if mentor['domain'] == mentee['interests']:
        score += weights['domain_match'] * 100
    else:
        score += weights['domain_match'] * 30

    # Skills alignment
    mentor_skills = set(str(mentor['skills']).split(','))
    mentee_interests = set([mentee['interests']])
    skill_overlap = len(mentor_skills.intersection(mentee_interests)) / max(1, len(mentor_skills.union(mentee_interests)))
    score += weights['skills_alignment'] * skill_overlap * 100

    # Experience matching
    if mentee.get('preferred_mentor_experience') == 'any':
        score += weights['experience_match'] * 100
    elif mentee.get('preferred_mentor_experience') == '2-5_years' and 2 <= mentor['experience_years'] <= 5:
        score += weights['experience_match'] * 100
    elif mentee.get('preferred_mentor_experience') == '5-10_years' and 5 <= mentor['experience_years'] <= 10:
        score += weights['experience_match'] * 100
    elif mentee.get('preferred_mentor_experience') == '10+_years' and mentor['experience_years'] >= 10:
        score += weights['experience_match'] * 100
    else:
        score += weights['experience_match'] * 50

    # Location preference
    if mentee.get('preferred_mentor_location') == 'any':
        score += weights['location_preference'] * 100
    elif mentee.get('preferred_mentor_location') == 'same_city' and mentor['location'] == mentee.get('location'):
        score += weights['location_preference'] * 100
    elif mentee.get('preferred_mentor_location') == 'same_university' and mentor['university'] == mentee.get('university'):
        score += weights['location_preference'] * 100
    elif mentee.get('preferred_mentor_location') == 'remote_only':
        score += weights['location_preference'] * 100
    else:
        score += weights['location_preference'] * 30

    # Communication style match
    if mentor.get('mentoring_style') == mentee.get('preferred_communication_style'):
        score += weights['communication_style'] * 100
    else:
        score += weights['communication_style'] * 60

    # Language compatibility
    mentor_langs = set(str(mentor.get('languages', 'English')).split(','))
    mentee_langs = set(str(mentee.get('languages', 'English')).split(','))
    lang_overlap = len(mentor_langs.intersection(mentee_langs)) > 0
    score += weights['language_match'] * (100 if lang_overlap else 50)

    # Availability match
    if mentor.get('availability_hours', 10) >= mentee.get('available_hours', 5):
        score += weights['availability_match'] * 100
    else:
        score += weights['availability_match'] * (mentor.get('availability_hours', 10) / max(1, mentee.get('available_hours', 5)) * 100)

    return round(score, 2)

# Generate matches
matching_data = []
num_matches = min(10000, len(df_mentors) * 2)  # Scale based on available mentors

for i in range(num_matches):
    mentor_idx = random.randint(0, len(df_mentors) - 1)
    mentee_idx = random.randint(0, len(df_mentees) - 1)

    mentor = df_mentors.iloc[mentor_idx].to_dict()
    mentee = df_mentees.iloc[mentee_idx].to_dict()

    compatibility_score = calculate_compatibility_score(mentor, mentee)

    matching_data.append({
        'match_id': f"MATCH_{i}",
        'mentor_id': mentor['mentor_id'],
        'mentee_id': mentee['student_id'],
        'compatibility_score': compatibility_score,
        'match_date': str(datetime.now() - timedelta(days=random.randint(0, 30))),
        'match_status': random.choice(['pending', 'accepted', 'rejected', 'completed']),
        'mentor_response_time_hours': random.randint(1, 72) if random.random() > 0.3 else None,
        'mentee_initiated': random.choice([True, False]),
        'match_reason': random.choice(['high_compatibility', 'specific_request', 'auto_suggestion', 'admin_assigned'])
    })

df_matching = pd.DataFrame(matching_data)
print(f"✅ Generated {len(df_matching)} mentorship matches")
print(f"📊 Average compatibility score: {df_matching['compatibility_score'].mean():.1f}/100")

🤝 Generating mentorship matching data...
✅ Generated 10000 mentorship matches
📊 Average compatibility score: 51.7/100


## 💬 Generate Communication Logs

In [12]:
# Generate communication logs
print("💬 Generating communication logs...")

communication_data = []
message_types = ["text", "file", "video_call", "voice_note", "schedule_request"]
message_status = ["sent", "delivered", "read", "replied"]

num_conversations = min(2000, len(df_matching) // 2)
for i in range(num_conversations):
    match = df_matching.sample(1).iloc[0]
    num_messages = random.randint(1, 20)

    for j in range(num_messages):
        sender_type = random.choice(["mentor", "mentee"])
        if sender_type == "mentor":
            sender_id = match['mentor_id']
            receiver_id = match['mentee_id']
        else:
            sender_id = match['mentee_id']
            receiver_id = match['mentor_id']

        communication_data.append({
            'message_id': f"MSG_{i}_{j}",
            'conversation_id': f"CONV_{i}",
            'match_id': match['match_id'],
            'sender_id': sender_id,
            'sender_type': sender_type,
            'receiver_id': receiver_id,
            'receiver_type': "mentee" if sender_type == "mentor" else "mentor",
            'message_type': random.choice(message_types),
            'message_content': f"Sample message content {j} in conversation {i}",
            'timestamp': str(datetime.now() - timedelta(days=random.randint(0, 30), hours=random.randint(0, 23))),
            'message_status': random.choice(message_status),
            'response_time_minutes': random.randint(1, 1440) if j > 0 else None,
            'attachment_count': random.randint(0, 3),
            'is_edited': random.choice([True, False]),
            'edit_timestamp': str(datetime.now() - timedelta(days=random.randint(0, 30))) if random.random() > 0.8 else None,
            'encryption_key': f"enc_key_{hashlib.md5(f'{i}_{j}'.encode()).hexdigest()[:16]}"
        })

df_communication = pd.DataFrame(communication_data)
print(f"✅ Generated {len(df_communication)} communication messages")

💬 Generating communication logs...
✅ Generated 20805 communication messages


## 🔌 Generate API Integration Logs

In [13]:
# Generate API integration logs
print("🔌 Generating API integration logs...")

api_data = []
api_endpoints = [
    "/mentorship/matches",
    "/mentorship/verify-document",
    "/mentorship/interview-score",
    "/mentorship/communication/send",
    "/mentorship/communication/history",
    "/mentorship/profiles/mentors",
    "/mentorship/profiles/mentees",
    "/mentorship/badges/calculate",
    "/mentorship/compatibility/calculate",
    "/mentorship/analytics/dashboard"
]

num_api_calls = 5000
for i in range(num_api_calls):
    endpoint = random.choice(api_endpoints)
    user_type = random.choice(["mentor", "mentee", "admin", "system"])

    api_data.append({
        'api_call_id': f"API_{i}",
        'endpoint': endpoint,
        'user_id': f"{random.choice(['M', 'S'])}{random.randint(0, min(9999, len(df_mentors)-1))}",
        'user_type': user_type,
        'request_timestamp': str(datetime.now() - timedelta(days=random.randint(0, 30), minutes=random.randint(0, 1440))),
        'request_method': random.choice(['GET', 'POST', 'PUT', 'DELETE']),
        'request_payload_size': random.randint(100, 10000),
        'response_status_code': random.choice([200, 201, 400, 401, 403, 404, 500]),
        'response_time_ms': random.randint(50, 2000),
        'response_payload_size': random.randint(50, 5000),
        'success': random.choice([True, False]),
        'error_message': f"Error description {i}" if random.random() > 0.8 else None,
        'ip_address': f"192.168.1.{random.randint(1, 254)}",
        'user_agent': "MentorshipApp/1.0",
        'api_version': "v1",
        'rate_limit_remaining': random.randint(0, 1000)
    })

df_api = pd.DataFrame(api_data)
print(f"✅ Generated {len(df_api)} API integration logs")

🔌 Generating API integration logs...
✅ Generated 5000 API integration logs


## 🛡️ Generate Privacy Workflow Data

In [14]:
# Generate privacy workflow data
print("🛡️ Generating privacy workflow data...")

privacy_data = []
document_types = ["CV", "Degree_Certificate", "Professional_Certificate", "Employment_Letter", "ID_Document"]

for i in range(min(len(df_mentors), 1000)):  # Limit to first 1000 mentors for performance
    mentor_id = df_mentors.iloc[i]['mentor_id']

    # Generate documents for this mentor
    num_documents = random.randint(2, 5)
    for doc_idx in range(num_documents):
        doc_id = f"DOC_{mentor_id}_{doc_idx}"
        doc_type = random.choice(document_types)

        # Processing steps
        processing_steps = [
            {
                'step': 'upload',
                'timestamp': str(datetime.now() - timedelta(days=random.randint(1, 365))),
                'action': 'encrypted_upload',
                'data_retained': 'encrypted_file_only'
            },
            {
                'step': 'ocr_processing',
                'timestamp': str(datetime.now() - timedelta(days=random.randint(1, 364), hours=1)),
                'action': 'metadata_extraction',
                'data_retained': 'extracted_text_only',
                'raw_file_deleted': True
            },
            {
                'step': 'verification',
                'timestamp': str(datetime.now() - timedelta(days=random.randint(1, 363), hours=2)),
                'action': 'automated_verification',
                'data_retained': 'verification_score_only'
            },
            {
                'step': 'cleanup',
                'timestamp': str(datetime.now() + timedelta(days=30)),  # Future deletion
                'action': 'permanent_deletion',
                'data_retained': 'verification_record_only'
            }
        ]

        for step in processing_steps:
            privacy_data.append({
                'privacy_id': f"PRIVACY_{mentor_id}_{doc_id}_{step['step']}",
                'mentor_id': mentor_id,
                'document_id': doc_id,
                'document_type': doc_type,
                'processing_step': step['step'],
                'timestamp': step['timestamp'],
                'action_taken': step['action'],
                'data_retained': step['data_retained'],
                'raw_file_deleted': step.get('raw_file_deleted', False),
                'encryption_method': 'AES-256-GCM',
                'compliance_check': random.choice(['GDPR', 'CCPA', 'PDPA']),
                'audit_log': True,
                'data_minimization_applied': True
            })

df_privacy = pd.DataFrame(privacy_data)
print(f"✅ Generated {len(df_privacy)} privacy workflow records")

🛡️ Generating privacy workflow data...
✅ Generated 14344 privacy workflow records


## 💾 Save All Datasets

In [15]:
# Save all datasets to Google Drive
print("💾 Saving all datasets to Google Drive...")

# Save enhanced existing datasets
df_mentors.to_csv(generated_datasets['matching'].replace('matching', 'enhanced_mentors'), index=False)
df_mentees.to_csv(generated_datasets['matching'].replace('matching', 'enhanced_mentees'), index=False)

# Save generated datasets
df_matching.to_csv(generated_datasets['matching'], index=False)
df_communication.to_csv(generated_datasets['communication'], index=False)
df_api.to_csv(generated_datasets['api'], index=False)
df_privacy.to_csv(generated_datasets['privacy'], index=False)

print("✅ All datasets saved successfully!")

print("\n📊 FINAL DATASET SUMMARY:")
print(f"  • Enhanced Mentors: {len(df_mentors):,} records")
print(f"  • Enhanced Mentees: {len(df_mentees):,} records")
print(f"  • Matching Data: {len(df_matching):,} records")
print(f"  • Communication Logs: {len(df_communication):,} records")
print(f"  • API Integration: {len(df_api):,} records")
print(f"  • Privacy Workflow: {len(df_privacy):,} records")

if df_alumni is not None:
    print(f"  • Original Alumni: {len(df_alumni):,} records (integrated into mentors)")

💾 Saving all datasets to Google Drive...
✅ All datasets saved successfully!

📊 FINAL DATASET SUMMARY:
  • Enhanced Mentors: 11,000 records
  • Enhanced Mentees: 5,000 records
  • Matching Data: 10,000 records
  • Communication Logs: 20,805 records
  • API Integration: 5,000 records
  • Privacy Workflow: 14,344 records
  • Original Alumni: 6,000 records (integrated into mentors)


## 🤖 Train ML Model with Your Data

In [16]:
# Train ML model using your real data
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

print("🤖 Training ML model with your real data...")

# Prepare training data
def prepare_training_data():
    training_data = []

    # Use existing matching data
    for _, match in df_matching.iterrows():
        try:
            mentor = df_mentors[df_mentors['mentor_id'] == match['mentor_id']].iloc[0]
            mentee = df_mentees[df_mentees['student_id'] == match['mentee_id']].iloc[0]

            features = {
                'domain_match': 1 if mentor['domain'] == mentee['interests'] else 0,
                'university_match': 1 if mentor['university'] == mentee['university'] else 0,
                'location_match': 1 if mentor['location'] == mentee.get('location', '') else 0,
                'mentor_experience': mentor['experience_years'],
                'mentor_verification_score': mentor['combined_verification_score'],
                'mentee_year': mentee['year_of_study'],
                'mentee_gpa': mentee['gpa'],
                'availability_match': 1 if mentor['availability_hours'] >= mentee['available_hours'] else 0,
                'communication_style_match': 1 if mentor.get('mentoring_style') == mentee.get('preferred_communication_style') else 0,
                'language_match': 1 if set(str(mentor.get('languages', 'English')).split(',')) & set(str(mentee.get('languages', 'English')).split(',')) else 0,
                'target': 1 if match['match_status'] == 'accepted' else 0
            }
            training_data.append(features)
        except:
            continue

    # Add negative examples
    for i in range(len(training_data)):
        mentor = df_mentors.sample(1).iloc[0]
        mentee = df_mentees.sample(1).iloc[0]

        features = {
            'domain_match': 1 if mentor['domain'] == mentee['interests'] else 0,
            'university_match': 1 if mentor['university'] == mentee['university'] else 0,
            'location_match': 1 if mentor['location'] == mentee.get('location', '') else 0,
            'mentor_experience': mentor['experience_years'],
            'mentor_verification_score': mentor['combined_verification_score'],
            'mentee_year': mentee['year_of_study'],
            'mentee_gpa': mentee['gpa'],
            'availability_match': 1 if mentor['availability_hours'] >= mentee['available_hours'] else 0,
            'communication_style_match': 1 if mentor.get('mentoring_style') == mentee.get('preferred_communication_style') else 0,
            'language_match': 1 if set(str(mentor.get('languages', 'English')).split(',')) & set(str(mentee.get('languages', 'English')).split(',')) else 0,
            'target': 0
        }
        training_data.append(features)

    return pd.DataFrame(training_data)

training_df = prepare_training_data()
print(f"📊 Training data prepared: {len(training_df)} samples")

# Split and train
X = training_df.drop('target', axis=1)
y = training_df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Evaluate
y_pred = rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"✅ Model trained successfully!")
print(f"📈 Accuracy: {accuracy:.2%}")
print(f"📊 Training samples: {len(X_train):,}")
print(f"📊 Test samples: {len(X_test):,}")

🤖 Training ML model with your real data...
📊 Training data prepared: 20000 samples
✅ Model trained successfully!
📈 Accuracy: 86.40%
📊 Training samples: 16,000
📊 Test samples: 4,000


## 🎯 Test Prediction System

In [17]:
# Test prediction system with your data
def predict_mentorship_success(mentor_id, mentee_id):
    """Predict success probability for mentor-mentee match"""
    try:
        mentor = df_mentors[df_mentors['mentor_id'] == mentor_id].iloc[0]
        mentee = df_mentees[df_mentees['student_id'] == mentee_id].iloc[0]

        features = pd.DataFrame([{
            'domain_match': 1 if mentor['domain'] == mentee['interests'] else 0,
            'university_match': 1 if mentor['university'] == mentee['university'] else 0,
            'location_match': 1 if mentor['location'] == mentee.get('location', '') else 0,
            'mentor_experience': mentor['experience_years'],
            'mentor_verification_score': mentor['combined_verification_score'],
            'mentee_year': mentee['year_of_study'],
            'mentee_gpa': mentee['gpa'],
            'availability_match': 1 if mentor['availability_hours'] >= mentee['available_hours'] else 0,
            'communication_style_match': 1 if mentor.get('mentoring_style') == mentee.get('preferred_communication_style') else 0,
            'language_match': 1 if set(str(mentor.get('languages', 'English')).split(',')) & set(str(mentee.get('languages', 'English')).split(',')) else 0
        }])

        success_prob = rf_model.predict_proba(features)[0][1]
        return round(success_prob * 100, 2)
    except Exception as e:
        return None

# Test with random pairs from your data
print("🎯 Testing prediction system with your data...")
print("-" * 60)

for i in range(5):
    mentor = df_mentors.sample(1).iloc[0]
    mentee = df_mentees.sample(1).iloc[0]

    success_prob = predict_mentorship_success(mentor['mentor_id'], mentee['student_id'])

    print(f"🎓 {mentor['name']} ↔ {mentee['name']}")
    print(f"   🏷️  {mentor['domain']} → {mentee['interests']}")
    print(f"   🎯 Success Probability: {success_prob}%")
    print()

🎯 Testing prediction system with your data...
------------------------------------------------------------
🎓 Mentor_4372 ↔ Student_1174
   🏷️  AI/ML → Networking
   🎯 Success Probability: 41.0%

🎓 Mentor_1574 ↔ Student_347
   🏷️  Networking → Software Engineering
   🎯 Success Probability: 8.0%

🎓 Mentor_4538 ↔ Student_3718
   🏷️  Cybersecurity → Business IT
   🎯 Success Probability: 1.0%

🎓 Mentor_711 ↔ Student_759
   🏷️  Data Science → Software Engineering
   🎯 Success Probability: 1.0%

🎓 Alumni_917 ↔ Student_210
   🏷️  Cybersecurity → Business IT
   🎯 Success Probability: 73.0%



## 📊 System Analytics

In [18]:
# Generate comprehensive analytics
print("📊 SYSTEM ANALYTICS & INSIGHTS")
print("=" * 50)

# Mentor Analytics
print("\n🎓 MENTOR ANALYTICS:")
print(f"  • Total Mentors: {len(df_mentors):,}")
print(f"  • Gold Badge: {len(df_mentors[df_mentors['verification_level'] == 'Gold']):,} ({len(df_mentors[df_mentors['verification_level'] == 'Gold'])/len(df_mentors)*100:.1f}%)")
print(f"  • Silver Badge: {len(df_mentors[df_mentors['verification_level'] == 'Silver']):,} ({len(df_mentors[df_mentors['verification_level'] == 'Silver'])/len(df_mentors)*100:.1f}%)")
print(f"  • Bronze Badge: {len(df_mentors[df_mentors['verification_level'] == 'Bronze']):,} ({len(df_mentors[df_mentors['verification_level'] == 'Bronze'])/len(df_mentors)*100:.1f}%)")
print(f"  • Average Experience: {df_mentors['experience_years'].mean():.1f} years")
print(f"  • Average Rating: {df_mentors['average_rating'].mean():.2f}/5.0")

if 'alumni_source' in df_mentors.columns:
    alumni_mentors = len(df_mentors[df_mentors['alumni_source'] == True])
    print(f"  • Alumni-based Mentors: {alumni_mentors:,} ({alumni_mentors/len(df_mentors)*100:.1f}%)")

# Mentee Analytics
print("\n👨‍🎓 MENTEE ANALYTICS:")
print(f"  • Total Mentees: {len(df_mentees):,}")
print(f"  • Average GPA: {df_mentees['gpa'].mean():.2f}")
print(f"  • Year Distribution:")
for year in sorted(df_mentees['year_of_study'].unique()):
    count = len(df_mentees[df_mentees['year_of_study'] == year])
    print(f"    - Year {year}: {count:,} ({count/len(df_mentees)*100:.1f}%)")

# Matching Analytics
print("\n🤝 MATCHING ANALYTICS:")
print(f"  • Total Matches: {len(df_matching):,}")
print(f"  • Accepted: {len(df_matching[df_matching['match_status'] == 'accepted']):,} ({len(df_matching[df_matching['match_status'] == 'accepted'])/len(df_matching)*100:.1f}%)")
print(f"  • Average Compatibility: {df_matching['compatibility_score'].mean():.1f}/100")
print(f"  • High Compatibility (>80): {len(df_matching[df_matching['compatibility_score'] > 80]):,}")

print(f"\n🎯 MODEL PERFORMANCE:")
print(f"  • Prediction Accuracy: {accuracy:.2%}")
print(f"  • Training Data Size: {len(training_df):,} samples")

print(f"\n✨ SYSTEM READY FOR PRODUCTION! ✨")

📊 SYSTEM ANALYTICS & INSIGHTS

🎓 MENTOR ANALYTICS:
  • Total Mentors: 11,000
  • Gold Badge: 3,673 (33.4%)
  • Silver Badge: 3,673 (33.4%)
  • Bronze Badge: 3,654 (33.2%)
  • Average Experience: 8.3 years
  • Average Rating: 4.25/5.0
  • Alumni-based Mentors: 6,000 (54.5%)

👨‍🎓 MENTEE ANALYTICS:
  • Total Mentees: 5,000
  • Average GPA: 3.24
  • Year Distribution:
    - Year 1: 1,265 (25.3%)
    - Year 2: 1,225 (24.5%)
    - Year 3: 1,272 (25.4%)
    - Year 4: 1,238 (24.8%)

🤝 MATCHING ANALYTICS:
  • Total Matches: 10,000
  • Accepted: 2,484 (24.8%)
  • Average Compatibility: 51.7/100
  • High Compatibility (>80): 0

🎯 MODEL PERFORMANCE:
  • Prediction Accuracy: 86.40%
  • Training Data Size: 20,000 samples

✨ SYSTEM READY FOR PRODUCTION! ✨


## 📥 Download Datasets

In [19]:
# Download all datasets
try:
    print("📥 Preparing datasets for download...")

    # Create zip file
    import zipfile
    import os

    zip_path = f'{DRIVE_PATH}complete_mentorship_system.zip'
    with zipfile.ZipFile(zip_path, 'w') as zipf:
        # Add all datasets
        datasets_to_zip = {
            'enhanced_mentors.csv': df_mentors,
            'enhanced_mentees.csv': df_mentees,
            'mentorship_matching.csv': df_matching,
            'communication_logs.csv': df_communication,
            'api_integration_logs.csv': df_api,
            'privacy_workflow.csv': df_privacy
        }

        for filename, dataframe in datasets_to_zip.items():
            temp_path = f'{DRIVE_PATH}{filename}'
            dataframe.to_csv(temp_path, index=False)
            zipf.write(temp_path, filename)

    # Download the zip
    files.download(zip_path)
    print("✅ Complete mentorship system downloaded!")

except Exception as e:
    print(f"⚠️ Download error: {e}")
    print("📁 Datasets saved in Google Drive for manual download")

📥 Preparing datasets for download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Complete mentorship system downloaded!
